# Knowledge Graph Construction Pipeline
## Strwythura-Adapted Academic ERKG

**Logging:** All steps log to `logs/kg_pipeline_<timestamp>.log` (DEBUG) and console (INFO).

| Cell | Strwythura Part | Purpose |
|---|---|---|
| 1 | Setup | Ontology + GLiNER + spaCy + Logging |
| 2 | Part 1 (ER) | Backbone from structured CSV |
| 3 | Part 3 (Parse) | spaCy + GLiNER + lemma-key entity store |
| 4 | Part 1+4 | Entity Resolution (3-layer) |
| 5 | Part 4 (HITL) | LLM Curation (validate + enrich) |
| 6 | Part 5 (Distil) | Distillation + Entity Embeddings |
| 7 | Part 6 (DB) | Neo4j + Weaviate ingestion |
| 8 | Verification | 10 Cypher test queries |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: Setup & Modular KG Imports (REMOTE MODE: 54.236.131.70)
# ══════════════════════════════════════════════════════════════════════
import os, sys, time
from pathlib import Path

# Inject project root to sys.path to allow imports from src.*
ROOT_DIR = str(Path(os.getcwd()).parent.parent)
if ROOT_DIR not in sys.path: sys.path.insert(0, ROOT_DIR)

import pandas as pd
import logging
from IPython.display import display

# Notebook Helpers (for compatibility with your existing cells)
def start_cell(name):
    print(f"\n{'='*60}\nSTART: {name}\n{'='*60}")

def end_cell(name, stats=None):
    print(f"\n{'─'*60}")
    if stats: 
        for k, v in stats.items(): print(f"  {k}: {v}")
    print(f"✅ {name} complete\n{'='*60}")

# KG Module Imports
from src.kg.config import NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD as NEO4J_PASS, WEAVIATE_HOST, WEAVIATE_PORT
from src.kg.services.kg_pipeline import KGPipeline
from src.kg.ontology import ONTOLOGY
from src.kg.utils import nlp # Auto-loaded spaCy

# Initialise Pipeline (Test Mode = 5 papers)
pipeline = KGPipeline(test_mode=True, clear_db=True)
logger = logging.getLogger('src.kg')

print(f"✅ Setup complete. Root: {ROOT_DIR}")
print(f"✅ Neo4j URI: {NEO4J_URI}")
print(f"✅ Weaviate: {WEAVIATE_HOST}:{WEAVIATE_PORT}")


✅ Setup complete. Root: c:\Users\rizky_11yf1be\Desktop\Tugas_Akhir
✅ Neo4j URI: bolt://54.236.131.70:7687
✅ Weaviate: 127.0.0.1:8081


In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2: Source Loading & Backbone Construction
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 2: Backbone Construction')
df_papers, df_dosen = pipeline.load_sources()
pipeline.build_backbone(df_papers, df_dosen)

display(pd.DataFrame(list(pipeline.nodes.values())).head())
end_cell('Cell 2: Backbone Construction', {'nodes': len(pipeline.nodes), 'edges': len(pipeline.edges)})



START: Cell 2: Backbone Construction
INFO     | ============================================================
INFO     | START: Step 1: Source Loading
INFO     | ============================================================
✅ Config: Loaded Dual-Proxy Credentials (Unlocker + SERP)
✅ Config: Loaded Credentials (Supabase: https://wfjzdhaaldwyiajbyzln.supabase.co)
✅ Supabase Client Connected to https://wfjzdhaaldwyiajbyzln.supabase.co
INFO     | Fetching data from Supabase tables: 'papers' and 'lecturers'...
INFO     | ────────────────────────────────────────────────────────────
INFO     |   Papers: 5
INFO     |   Dosen: 129
INFO     | ✅ Step 1: Source Loading completed in 5.2s
INFO     | ============================================================

INFO     | ============================================================
INFO     | START: Step 2: Backbone Construction
INFO     | ============================================================
INFO     | Loading GLiNER model: urchade/gliner_larg

c:\Users\rizky_11yf1be\Desktop\Tugas_Akhir\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

INFO     | ✅ GLiNER model loaded: urchade/gliner_large-v2.1
INFO     | Dosen backbone: 129 dosen registered
INFO     | Filtered: 5 with abstracts, sampled 5
INFO     | Backbone: 5 papers, 129 dosen, 0 external skipped, 13 keywords | Nodes: 176, Edges: 162
INFO     | ────────────────────────────────────────────────────────────
INFO     |   Total nodes: 176
INFO     |   Total edges: 162
INFO     | ✅ Step 2: Backbone Construction completed in 15.3s
INFO     | ============================================================



,_label,name,prodi,scholar_id,nip,nidn,title,year,url,journal,value
0,Dosen,Yuni Yamasari,S2 Informatika,hn5jrnAAAAAJ,197506022003122001,0002067504,NaN,NaN,NaN,NaN,NaN
1,ProgramStudi,S2 Informatika,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dosen,Agus Prihanto,S1 Teknik Informatika,-_VNTSIAAAAJ,197908062006041001,0006087903,NaN,NaN,NaN,NaN,NaN
3,ProgramStudi,S1 Teknik Informatika,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dosen,Ricky Eka Putra,S2 Informatika,cPj-iOIAAAAJ,198701162018031001,0716018704,NaN,NaN,NaN,NaN,NaN



────────────────────────────────────────────────────────────
  nodes: 176
  edges: 162
✅ Cell 2: Backbone Construction complete


In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: NER Extraction
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 3: NER Extraction')
pipeline.extract_entities()

ent_list = [{"text": e['text'], "label": e['label'], "src": e['source']} 
            for i, e in enumerate(pipeline.entity_store.entities.values()) if i < 10]
display(pd.DataFrame(ent_list))
end_cell('Cell 3: NER Extraction', pipeline.entity_store.stats)



START: Cell 3: NER Extraction
INFO     | ============================================================
INFO     | START: Step 3: NER Extraction
INFO     | ============================================================
INFO     |   Parsed 5/5 papers | unique entities: 20
INFO     | ────────────────────────────────────────────────────────────
INFO     |   unique_entities: 20
INFO     |   from_ner: 0
INFO     |   from_title_regex: 20
INFO     |   from_csv_keywords: 0
INFO     | ✅ Step 3: NER Extraction completed in 19.3s
INFO     | ============================================================



,text,label,src
0,Pengembangan Sistem Rekomendasi Film Berbasis ...,Method,1
1,Implementasi Algoritma C5,Method,1
2,Pada Klasifikasi Status Gizi Balita Di Kecamat...,Method,1
3,WEB,Method,1
4,CV,Method,1
5,SKM,Method,1
6,INDONESIA,Method,1
7,Perancangan Dan Implementasi Sistem Penyimpana...,Method,1
8,SKM INDONESIA,Method,1
9,SMPN,Method,1



────────────────────────────────────────────────────────────
  unique_entities: 20
  from_ner: 0
  from_title_regex: 20
  from_csv_keywords: 0
✅ Cell 3: NER Extraction complete


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4: Entity Resolution
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 4: Entity Resolution')
pipeline.resolve_entities()
end_cell('Cell 4: Entity Resolution', {'Alias mappings': len(pipeline.alias_map)})



START: Cell 4: Entity Resolution
INFO     | ============================================================
INFO     | START: Step 4: Entity Resolution
INFO     | ============================================================
INFO     | ✅ GroqClient initialised (model=llama-3.3-70b-versatile)
INFO     | Layer 2 (Regex): 0 abbreviation aliases from 0 matches
INFO     | Layer 3 (LLM): 1 synonym clusters from 1 batches
INFO     | Total alias mappings: 1
INFO     | ────────────────────────────────────────────────────────────
INFO     |   Total alias mappings: 1
INFO     |   Entity references merged: 1
INFO     | ✅ Step 4: Entity Resolution completed in 2.6s
INFO     | ============================================================


────────────────────────────────────────────────────────────
  Alias mappings: 1
✅ Cell 4: Entity Resolution complete


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: LLM Curation
# ══════════════════════════════════════════════════════════════════════
start_cell('Cell 5: LLM Curation')
pipeline.curate_entities()
end_cell('Cell 5: LLM Curation', {'Curated entities': len(pipeline.entity_vdb)})



START: Cell 5: LLM Curation
INFO     | ============================================================
INFO     | START: Step 5: LLM Curation
INFO     | ============================================================
INFO     | ✅ GroqClient initialised (model=llama-3.3-70b-versatile)
INFO     | Valid entity labels for curation: {'Innovation', 'Tool', 'Task', 'Problem', 'Model', 'Metric', 'Field', 'Method', 'Dataset'}
INFO     | LLM Curation complete: 30 entities, 9 relations, 0 errors, 0 skipped
INFO     | ────────────────────────────────────────────────────────────
INFO     |   Curated entities: 30
INFO     |   Curated relations: 9
INFO     |   Content keywords: 5
INFO     |   Total nodes now: 206
INFO     |   Total edges now: 201
INFO     | ✅ Step 5: LLM Curation completed in 11.8s
INFO     | ============================================================


────────────────────────────────────────────────────────────
  Curated entities: 30
✅ Cell 5: LLM Curation complete


In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6: Distillation & Entity Embeddings (Strwythura Part 5)
# ══════════════════════════════════════════════════════════════════════
import gensim
start_cell('Cell 6: Distillation & Embeddings')

# Access via pipeline object
entity_store_entities = pipeline.entity_store.entities
entity_sequences = list(pipeline.extracted_entities.values())
nodes = pipeline.nodes
edges = pipeline.edges

high_centrality = {lk for lk, e in entity_store_entities.items() if e['count'] >= 2}
low_centrality = {lk for lk, e in entity_store_entities.items() if e['count'] < 2}
logger.info(f'Centrality filter: {len(high_centrality)} high (count>=2), {len(low_centrality)} low (count<2)')

w2v_sentences = []
for seq in entity_sequences:
    if len(seq) >= 2:
        w2v_sentences.append([str(uid) for uid in seq])

logger.info(f'Word2Vec training sentences: {len(w2v_sentences)}')

if w2v_sentences:
    w2v_model = gensim.models.Word2Vec(
        sentences=w2v_sentences,
        vector_size=32,
        window=max(len(s) for s in w2v_sentences),
        min_count=1,
        sg=1,
    )
    w2v_model.save('file_tabulars/entity_embeddings.w2v')
    logger.info(f'Entity embeddings trained: {len(w2v_model.wv)} vectors, dim={w2v_model.wv.vector_size}')
    uid_to_lk = {str(e['uid']): lk for lk, e in entity_store_entities.items()}
    for test_uid in list(w2v_model.wv.index_to_key)[:3]:
        similar = w2v_model.wv.most_similar(test_uid, topn=3)
        test_name = entity_store_entities.get(uid_to_lk.get(test_uid, ''), {}).get('text', '?')
        sim_names = [(entity_store_entities.get(uid_to_lk.get(s[0], ''), {}).get('text', '?'), round(s[1], 3)) for s in similar]
        logger.info(f'  Similar to "{test_name}": {sim_names}')
else:
    logger.warning('Not enough entity sequences for Word2Vec training')

end_cell('Cell 6: Distillation & Embeddings', {
    'High centrality entities': len(high_centrality),
    'Low centrality (filtered)': len(low_centrality),
    'W2V sentences': len(w2v_sentences),
    'Total nodes': len(nodes), 'Total edges': len(edges)
})



START: Cell 6: Distillation & Embeddings
INFO     | Centrality filter: 1 high (count>=2), 19 low (count<2)
INFO     | Word2Vec training sentences: 4
INFO     | Entity embeddings trained: 18 vectors, dim=32
INFO     |   Similar to "?": [('?', 0.369), ('?', 0.368), ('?', 0.217)]
INFO     |   Similar to "?": [('?', 0.318), ('?', 0.305), ('?', 0.244)]
INFO     |   Similar to "?": [('?', 0.326), ('?', 0.312), ('?', 0.294)]

────────────────────────────────────────────────────────────
  High centrality entities: 1
  Low centrality (filtered): 19
  W2V sentences: 4
  Total nodes: 206
  Total edges: 201
✅ Cell 6: Distillation & Embeddings complete


In [8]:
start_cell('Cell 7: Database Ingestion')
pipeline.ingest_databases()
end_cell('Cell 7: Database Ingestion')



START: Cell 7: Database Ingestion
INFO     | ============================================================
INFO     | START: Step 6: Database Ingestion
INFO     | ============================================================
INFO     | ✅ Neo4jKGWriter connected to bolt://54.236.131.70:7687 (db=infokom-unesa)
INFO     |   Neo4j: all existing data deleted
INFO     |   Neo4j: 15 uniqueness constraints ensured
INFO     | Ingesting 206 nodes to Neo4j...
INFO     |   Neo4j nodes ingested in 64.2s (errors: 0)
INFO     | Ingesting 201 edges to Neo4j...
INFO     |   Neo4j edges ingested in 85.8s (errors: 0, skipped: 0)
INFO     |   COLLABORATES_WITH edges derived: 15
INFO     | 
INFO     | KG CONSTRUCTION PIPELINE COMPLETE!
INFO     | ============================================================
INFO     | Neo4j: 206 nodes, 216 edges
INFO     | --- Node Distribution ---
INFO     |   Dosen               : 127
INFO     |   ProgramStudi        : 24
INFO     |   Keyword             : 13
INFO     |   

c:\Users\rizky_11yf1be\Desktop\Tugas_Akhir\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


INFO     |   Created collection: EntityEmbedding (vectorizer=text2vec-huggingface, model=sentence-transformers/all-MiniLM-L6-v2)
INFO     |   Deleted existing collection: RelationshipEmbedding
INFO     |   Created collection: RelationshipEmbedding (vectorizer=text2vec-huggingface, model=sentence-transformers/all-MiniLM-L6-v2)
INFO     |   Deleted existing collection: ContentKeyword
INFO     |   Created collection: ContentKeyword (vectorizer=text2vec-huggingface, model=sentence-transformers/all-MiniLM-L6-v2)
INFO     |   Deleted existing collection: PaperChunk
INFO     |   Created collection: PaperChunk (vectorizer=text2vec-huggingface, model=sentence-transformers/all-MiniLM-L6-v2)
INFO     | Ingesting to Weaviate (batched)...
INFO     |   EntityEmbedding: 30 objects (batch errors: 0)
INFO     |   RelationshipEmbedding: 9 objects (batch errors: 0)
INFO     |   ContentKeyword: 5 objects (batch errors: 0)
INFO     |   PaperChunk: 5 objects (batch errors: 0)
INFO     | Weaviate client clos

In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8: Verification Queries
# ══════════════════════════════════════════════════════════════════════
from neo4j import GraphDatabase
start_cell('Cell 8: Verification Queries')

# Membuka ulang koneksi database spesifik untuk testing ini
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

queries = [
    ('1. Metode Terpopuler', 'MATCH (p:Paper)-[:HAS_METHOD]->(m:Method) RETURN m.name AS Metode, COUNT(p) AS Jumlah ORDER BY Jumlah DESC LIMIT 10'),
    ('2. Paper Mirip (Metode Sama)', 'MATCH (p1:Paper)-[:HAS_METHOD]->(m:Method)<-[:HAS_METHOD]-(p2:Paper) WHERE p1 <> p2 RETURN p1.title AS Paper1, m.name AS Metode, p2.title AS Paper2 LIMIT 10'),
    ('3. Topik per Prodi', 'MATCH (ps:ProgramStudi)<-[:MEMBER_OF]-(d:Dosen)-[:WRITES]->(p:Paper)-[:IN_FIELD]->(f:Field) RETURN ps.name AS Prodi, f.name AS Field, COUNT(p) AS Total ORDER BY Total DESC LIMIT 10'),
    ('4. Kolaborasi Antar Dosen', 'MATCH (d1:Dosen)-[:COLLABORATES_WITH]->(d2:Dosen) RETURN d1.name AS Dosen1, d2.name AS Dosen2, d1.prodi AS Prodi1, d2.prodi AS Prodi2 LIMIT 10'),
    ('5. Tren Model per Tahun', 'MATCH (p:Paper)-[:HAS_MODEL]->(m:Model) MATCH (p)-[:PUBLISHED_YEAR]->(y:Year) RETURN y.value AS Tahun, m.name AS Model, COUNT(p) AS Jumlah ORDER BY Tahun DESC, Jumlah DESC LIMIT 10'),
    ('6. Dosen Produktif & Spesialisasi', 'MATCH (d:Dosen)-[:WRITES]->(p:Paper)-[:IN_FIELD]->(f:Field) RETURN d.name AS Dosen, f.name AS Topik, COUNT(p) AS Total ORDER BY Total DESC LIMIT 10'),
    ('7. Problem vs Tool', 'MATCH (t:Tool)<-[:HAS_TOOL]-(p:Paper)-[:ADDRESSES]->(pr:Problem) RETURN pr.name AS Masalah, t.name AS Tool, COUNT(p) AS Kasus ORDER BY Kasus DESC LIMIT 10'),
    ('8. Metrik per Field', 'MATCH (f:Field)<-[:IN_FIELD]-(p:Paper)-[:HAS_METRIC]->(m:Metric) RETURN f.name AS Field, m.name AS Metrik, COUNT(p) AS Freq ORDER BY Freq DESC LIMIT 10'),
    ('9. Paper Kompleks', 'MATCH (p:Paper)-[:HAS_METHOD|HAS_MODEL]->(e) WITH p, COUNT(e) AS K, COLLECT(e.name) AS List WHERE K>1 RETURN p.title AS Paper, K, List ORDER BY K DESC LIMIT 5'),
    ('10. Pencarian Pakar Sebidang', 'MATCH (d1:Dosen)-[:WRITES]->(p1:Paper)-[:ADDRESSES]->(pr:Problem)<-[:ADDRESSES]-(p2:Paper)<-[:WRITES]-(d2:Dosen) WHERE d1<>d2 AND elementId(d1)<elementId(d2) RETURN pr.name AS Problem, d1.name AS Peneliti1, d2.name AS Peneliti2 LIMIT 10'),
]

passed, failed = 0, 0
for title, q in queries:
    logger.info(f'\n--- {title} ---')
    logger.debug(f'Query: {q}')
    with driver.session(database="infokom-unesa") as s:
        try:
            result = s.run(q).data()
            if result:
                passed += 1
                logger.info(f'  ✅ {len(result)} rows returned')
                display(pd.DataFrame(result))
            else:
                failed += 1
                logger.warning(f'  ⚠️ No results')
        except Exception as e:
            failed += 1
            logger.error(f'  ❌ Query error: {type(e).__name__}: {e}')

driver.close()

end_cell('Cell 8: Verification Queries', {
    'Queries passed': passed, 
    'Queries with no results': failed,
    'Status': 'Successfully executed independently'
})
logger.info("Verification queries complete. See terminal/file logs for details.")



START: Cell 8: Verification Queries
INFO     | 
--- 1. Metode Terpopuler ---
INFO     |   ✅ 8 rows returned


,Metode,Jumlah
0,Collaborative Filtering,1
1,Quantitative experimentation methods,1
2,User testing assessment,1
3,Algoritma C5.0,1
4,Black Box Testing,1
5,PjBL,1
6,ADDIE,1
7,R&D,1


INFO     | 
--- 2. Paper Mirip (Metode Sama) ---
WARNING  |   ⚠️ No results
INFO     | 
--- 3. Topik per Prodi ---


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: IN_FIELD)} {position: line: 1, column: 72, offset: 71} for query: 'MATCH (ps:ProgramStudi)<-[:MEMBER_OF]-(d:Dosen)-[:WRITES]->(p:Paper)-[:IN_FIELD]->(f:Field) RETURN ps.name AS Prodi, f.name AS Field, COUNT(p) AS Total ORDER BY Total DESC LIMIT 10'


WARNING  |   ⚠️ No results
INFO     | 
--- 4. Kolaborasi Antar Dosen ---
INFO     |   ✅ 10 rows returned


...

WARNING  |   ⚠️ No results
INFO     | 
--- 7. Problem vs Tool ---
WARNING  |   ⚠️ No results
INFO     | 
--- 8. Metrik per Field ---


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: IN_FIELD)} {position: line: 1, column: 20, offset: 19} for query: 'MATCH (f:Field)<-[:IN_FIELD]-(p:Paper)-[:HAS_METRIC]->(m:Metric) RETURN f.name AS Field, m.name AS Metrik, COUNT(p) AS Freq ORDER BY Freq DESC LIMIT 10'


WARNING  |   ⚠️ No results
INFO     | 
--- 9. Paper Kompleks ---
INFO     |   ✅ 3 rows returned


,Paper,K,List
0,Pengembangan Sistem Rekomendasi Film Berbasis ...,3,"[User testing assessment, Quantitative experim..."
1,Pengembangan E-Modul Berbasis Web Dengan Model...,3,"[R&D, ADDIE, PjBL]"
2,Implementasi Algoritma C5. 0 Pada Klasifikasi ...,2,"[Decision Tree C5.0, Algoritma C5.0]"


INFO     | 
--- 10. Pencarian Pakar Sebidang ---
WARNING  |   ⚠️ No results

────────────────────────────────────────────────────────────
  Queries passed: 4
  Queries with no results: 6
  Status: Successfully executed independently
✅ Cell 8: Verification Queries complete
INFO     | Verification queries complete. See terminal/file logs for details.
